In [3]:
######################################
# Quantum teleportation
######################################

"""
A simple simulation for quantum teleportation. The simplest case with three qubits. 
Alice and Bob share an entangled state. There is also an unknown "mystery" qubit that is with Alice. 
Alice tries to send information about the mystery qubit to Bob without destroying it. Bob tries to reconstruct the mystery qubit with the information she got from Alice.
"""
# From the Qiskit library itself we need the following modules
from qiskit import QuantumCircuit, ClassicalRegister, QuantumRegister

import numpy as np

# With an explicit seed, the experiment can be reproduced
rng =np.random.default_rng(30)

# To implement quantum teleportation, we need three qubits. Two for Alice and one for Bob

A=QuantumRegister(1, "A")  # Alice
B= QuantumRegister(1, "B")  # Bob
C= QuantumRegister(1, "C")  # Mystery qubit-Alice

# We also need classical registers. These classical bits contain the information about the actual measurements
AC_bit =ClassicalRegister(2, "AC_cl")  # Contains information about qubits A and C
B_bit= ClassicalRegister(1, "B_cl")  # Contains information about qubit B

# Form the circuit. It consists of the qubits and bits
circuit=QuantumCircuit(A, B, C, AC_bit, B_bit)

# Here we need to create the entagled qubit. In our case we create bell state with the basic gate-operations; Hadamard and CNOT.
circuit.h(A)
circuit.cx(A, B)

# These are for the mystery qubit. We essentially create a random qubit with these amplitudes
theta= rng.uniform(0.0, 1.0) * np.pi
varphi=rng.uniform(0.0, 2.0) * np.pi

# Create the mystery qubit
circuit.u(theta, varphi, 0.0, C)

# Combine the mystery qubit with the Alice's bell state part, so with qubit A
circuit.cx(C, A)
# Use Hadamard on mystery qubit
circuit.h(C)

# Make the measurements for the qubit A and C. The information is stored into the bits which were defined earlier
# Alice's measurement destroys the mystery qubit C --> no-cloning theorem. The quantum information teleports to Bob

circuit.measure(A, AC_bit[1])
circuit.measure(C, AC_bit[0])

# This part adds the possible operations that need to be done by Bob
# So depending on Alice's measurement outcome, Bob uses X, Z or both gates to recover the state
with circuit.if_test((AC_bit[1], 1)):
    circuit.x(B)
with circuit.if_test((AC_bit[0], 1)):
    circuit.z(B)

# In the end measure Bob's qubit
circuit.measure(B, B_bit)


# Here we just simulate the what the end result would be with our specific mystery qubit
# We use Aer as the simulator
from qiskit_aer import AerSimulator
from qiskit import transpile

simulator= AerSimulator()
compiled=transpile(circuit, simulator)#Transpiling prepares the gates in a way that they can be used on a backend, this can be changed so that real hardware can be used!

# Run the simulator
to_do=simulator.run(compiled, shots=2048)
result=to_do.result()

# To count the outcomes, and print them
counts=result.get_counts()
print(counts)

# Verify the teleportation, Bob's qubit should match the original mystery qubit
b0,b1= 0, 0
for outcome, count in counts.items():
    b_bit_val=outcome.split(' ')[0]   
    if b_bit_val=='0': 
        b0 += count
    else:
        b1 += count

total=b0 + b1
print(f"\nMystery qubit: theta={theta:.4f}  varphi={varphi:.4f}")
print(f"Expected:  |0> probability: {np.cos(theta/2)**2:.4f}")
print(f"Measured:  |0> probability: {b0/total:.4f}")
print(f"Expected:  |1> probability: {np.sin(theta/2)**2:.4f}")
print(f"Measured:  |1> probability: {b1/total:.4f}"  )

{'0 10': 437, '0 11': 480, '0 01': 435, '0 00': 423, '1 00': 74, '1 01': 52, '1 11': 71, '1 10': 76}

Mystery qubit: theta=0.7408  varphi=2.6962
Expected:  |0> probability: 0.8690
Measured:  |0> probability: 0.8667
Expected:  |1> probability: 0.1310
Measured:  |1> probability: 0.1333


As it can be seen, the measured probabilities are very close to the mystery qubit. This means that the teleportation was a success! The error could be less if more shots were added.